In [4]:
import os
import torch
import torch.nn as nn 
import numpy as np
import pyexr
from torch.utils.data import Dataset, DataLoader

In [2]:
class BaselineHolographyDataset(Dataset):
    def __init__(self, root_dir):
        self.img_dir = os.path.join(root_dir, "img")
        self.depth_dir = os.path.join(root_dir, "depth")
        self.amp_dir = os.path.join(root_dir, "amp")
        self.phs_dir = os.path.join(root_dir, "phs")
        
        # Get indices
        self.indices = sorted([f.replace(".exr", "") for f in os.listdir(self.img_dir) if f.endswith(".exr")])

    def load_exr(self, path):
        file = pyexr.open(path)
        return file.get().astype(np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        
        # 1. Load raw RGB and Depth
        rgb_raw = self.load_exr(os.path.join(self.img_dir, f"{i}.exr"))
        depth_raw = self.load_exr(os.path.join(self.depth_dir, f"{i}.exr"))
        
        # 2. Prepare 4-Channel Input [RGB (3) + Depth (1)]
        # RGB to [3, 512, 512]
        rgb = torch.from_numpy(rgb_raw).permute(2, 0, 1) if rgb_raw.ndim == 3 else torch.from_numpy(rgb_raw).unsqueeze(0)
        # Depth to [1, 512, 512]
        depth = torch.from_numpy(depth_raw).unsqueeze(0) if depth_raw.ndim == 2 else torch.from_numpy(depth_raw).permute(2, 0, 1)[:1]
        
        x = torch.cat([rgb, depth], dim=0) # [4, 512, 512]
        
        # 3. Prepare 2-Channel Target [Amp, Phase]
        amp = torch.from_numpy(self.load_exr(os.path.join(self.amp_dir, f"{i}.exr")))
        phs = torch.from_numpy(self.load_exr(os.path.join(self.phs_dir, f"{i}.exr")))
        
        if amp.ndim == 3: amp = amp[:, :, 0]
        if phs.ndim == 3: phs = phs[:, :, 0]
        y = torch.cat([amp.unsqueeze(0), phs.unsqueeze(0)], dim=0) # [2, 512, 512]

In [3]:
# Initialize Baseline Data
data_root = r"C:\Users\Kai Kumano\workspace\CGH-depth\dataset\KOREATECH-CGH-512-3.6Mu"
train_baseline = BaselineHolographyDataset(os.path.join(data_root, "train"))
train_loader = DataLoader(train_baseline, batch_size=4, shuffle=True)

In [6]:

# Change this part
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class SimpleUNetBaseline(nn.Module):
    def __init__(self, in_channels=4, out_channels=2):
        super(SimpleUNetBaseline, self).__init__()

        # Encoder
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        u1 = self.up1(x5); u1 = torch.cat([u1, x4], dim=1); u1 = self.conv_up1(u1)
        u2 = self.up2(u1); u2 = torch.cat([u2, x3], dim=1); u2 = self.conv_up2(u2)
        u3 = self.up3(u2); u3 = torch.cat([u3, x2], dim=1); u3 = self.conv_up3(u3)
        u4 = self.up4(u3); u4 = torch.cat([u4, x1], dim=1); u4 = self.conv_up4(u4)
        return self.outc(u4)
    
model_baseline = SimpleUNetBaseline(in_channels=4, out_channels=2).to(device)
print(f"Baseline Model initialized on: {device}")

Baseline Model initialized on: cuda
